# 03 - Stable Video Diffusion + LoRA fine-tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matu1003/Makeitalive/blob/main/notebooks/03_svd_lora.ipynb)

[Stable Video Diffusion](https://huggingface.co/stabilityai/stable-video-diffusion-img2vid) (SVD) is a pretrained
image-to-video diffusion model: from one picture it generates 14 frames, and it can **create new content**,
which warping cannot. Out of the box, its motion on landscapes is short and hesitant, so we fine-tuned it on drone footage.

This notebook covers our first strategy, **LoRA**: the whole model is frozen and small low-rank adapters are trained
on the **temporal attention layers only** (about 3.3M parameters), the layers responsible for the consistency of motion across frames.

LoRA turned out to be too constrained: it only learned slight parallax (final loss 0.51). Our final results come from a
**full fine-tuning of the temporal layers** with [SVD_Xtend](https://github.com/pixeli99/SVD_Xtend)
(spatial layers frozen, lr 1e-5, 5000 steps, final loss 0.16), whose training code is not part of this repository.
See the [report](../docs/report.pdf) for the comparison.

Requirements: an A100-class GPU, the `svd` extra (`uv sync --extra svd`) and a Hugging Face account to download the SVD weights.

In [ ]:
import sys
from pathlib import Path

# On Colab, clone the repo and install it; locally, run `uv sync` first.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/matu1003/Makeitalive.git
    %cd Makeitalive
    !pip install -q -e .[svd]
    REPO_ROOT = Path.cwd()
else:
    REPO_ROOT = Path.cwd().parent

DATA_DIR = REPO_ROOT / "data"
CKPT_DIR = REPO_ROOT / "checkpoints"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

from types import SimpleNamespace
from IPython.display import Video
import matplotlib.pyplot as plt
from PIL import Image
from huggingface_hub import login

from data.video_utils import save_first_frame
from svd_lora.make_dataset_svd import extract_clips
from svd_lora import train_svd_lora

login()  # Hugging Face token with "read" access

## 1. Experiments

Three LoRA configurations were tried. `frame_gap` controls how many source frames are skipped between two frames of a clip:
with 25, a 14-frame clip covers ~15 s of drone footage (strong, time-lapse-like motion); with 12, the motion is gentler.
The dataset used for the final SVD_Xtend run was extracted with `frame_gap=12` (2,374 clips kept).

In [ ]:
EXPERIMENTS = {
    # 2000 clips, rank 16
    "base":        dict(max_clips=2000, frame_gap=25, interval=4.0, lora_rank=16, epochs=3),
    # more clips, sampled densely
    "5k_clips":    dict(max_clips=5000, frame_gap=25, interval=0.5, lora_rank=16, epochs=3),
    # shorter motion span, larger LoRA, longer training
    "5k_rank32":   dict(max_clips=5000, frame_gap=12, interval=7.0, lora_rank=32, epochs=14),
}

EXPERIMENT = "5k_rank32"
cfg = EXPERIMENTS[EXPERIMENT]
CLIPS_DIR = DATA_DIR / f"svd_{EXPERIMENT}"
RUN_DIR = CKPT_DIR / f"svd_lora_{EXPERIMENT}"

## 2. Extract the clips

Clips of 14 frames at 512x512 are extracted from the same drone compilation. A clip is dropped if it contains a
hard cut, if it is too static (`motion_min`) or if its motion is too violent (`motion_max`, usually glitches or transitions).

In [ ]:
extract_clips(
    youtube_url="https://www.youtube.com/watch?v=AKeUssuu3Is",
    output_dir=str(CLIPS_DIR),
    clip_len=14,
    fps=7,
    target_size=512,
    sample_every_n_seconds=cfg["interval"],
    frame_gap=cfg["frame_gap"],
    max_clips=cfg["max_clips"],
    motion_min=3.0,
    motion_max=40.0,
)

In [ ]:
clip = sorted(CLIPS_DIR.glob("clip_*"))[0]
frames = sorted(clip.glob("frame_*.jpg"))
fig, axes = plt.subplots(1, 7, figsize=(21, 3))
for ax, f in zip(axes, frames[::2]):
    ax.imshow(Image.open(f))
    ax.set_title(f.stem)
    ax.axis("off")
plt.suptitle(clip.name)
plt.show()

## 3. LoRA fine-tuning

Each step encodes the clip into the VAE latent space, adds noise with the EDM schedule used by SVD, and trains the
LoRA adapters to denoise it, conditioned on the first frame (CLIP image embedding + VAE latent).
Weights are saved to `<run>/lora_best` and `<run>/lora_latest`.

In [ ]:
train_svd_lora.train(SimpleNamespace(
    data_dir=str(CLIPS_DIR),
    output_dir=str(RUN_DIR),
    clip_len=14,
    size=512,
    epochs=cfg["epochs"],
    batch_size=1,
    lr=1e-4,
    lora_rank=cfg["lora_rank"],
    fps_id=7,
    motion_bucket_id=127,
))

## 4. Pretrained SVD vs SVD + LoRA

The same picture is animated by the pretrained model (`lora_dir=None`) and by the LoRA fine-tuned one.

In [ ]:
# Input picture: the first frame of the pretrained SVD result video, i.e. the original still image
TEST_IMAGE = save_first_frame(REPO_ROOT / "assets" / "videos" / "svd" / "SVD_Not_Trained.mp4", OUTPUT_DIR / "svd_input.jpg")
lora_best = sorted(RUN_DIR.glob("run_*/lora_best"))[-1]

train_svd_lora.run_inference(None, str(TEST_IMAGE), str(OUTPUT_DIR / "svd_baseline.mp4"))
train_svd_lora.run_inference(str(lora_best), str(TEST_IMAGE), str(OUTPUT_DIR / f"svd_lora_{EXPERIMENT}.mp4"))

In [ ]:
display(Video(str(OUTPUT_DIR / "svd_baseline.mp4"), embed=True, width=480))
display(Video(str(OUTPUT_DIR / f"svd_lora_{EXPERIMENT}.mp4"), embed=True, width=480))

## 5. Results

| Pretrained SVD | Temporal fine-tuning (SVD_Xtend) |
|:---:|:---:|
| <img src="../assets/gifs/svd_not_trained.gif" width="320"> | <img src="../assets/gifs/svd_final_2.gif" width="320"> |

<img src="../assets/gifs/svd_final_1.gif" width="640">

The LoRA runs of this notebook only add slight parallax to the pretrained behaviour. Fully fine-tuning the temporal layers
with SVD_Xtend produces smooth drone-like camera motion with convincing parallax and generates the parts of the scene revealed
by the movement. The raw 7 fps output is slowed down to 2 fps and interpolated to 24 fps with `ffmpeg minterpolate`.